# Omni NCM — Colab end-to-end: clone, check, train both arms, eval

One process streams the Hub once and **both arms step the identical batch** (`strict` level-parallel
vs flat `ar`), so the matched control is structural — no shard corpus, no produce step.

`1875 steps x 16 = 30,000 samples`, seen exactly once, `504` packed positions each -> **0.18
positions per parameter** at 82.88M. That exercises the machinery; it is *not* a strict-vs-AR
verdict. Multiply 30,000 x 504 and divide by the `pos_per_s` cell 5 prints to get your own wall
clock, since the stream rate depends on Colab's network, not on the GPU.

In [ ]:
# ---- env: dotenv first (local .env), then Colab secrets, then the process environment ----
import os, sys, subprocess

try:
    from dotenv import load_dotenv
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "python-dotenv"], check=True)
    from dotenv import load_dotenv
load_dotenv()                       # no-op in Colab; picks up .env when run locally

def _secret(name):
    v = os.environ.get(name)
    if v and v.strip():
        return v.strip()
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v and v.strip():
            return v.strip()
    except Exception:
        pass
    return ""

GITHUB_TOKEN, HF_TOKEN = _secret("GITHUB_TOKEN"), _secret("HF_TOKEN")
assert GITHUB_TOKEN, "set GITHUB_TOKEN (Colab secret, .env, or env) - the repo is private"
os.environ["HF_TOKEN"] = HF_TOKEN
print("token length", len(GITHUB_TOKEN), "| hf token", "set" if HF_TOKEN else "MISSING")
print("python", sys.executable)

In [ ]:
# ---- clone or update, then install the extra readers ----
REPO, PKG = "adi274903/ncm-omni", "/content/ncm_omni"

askpass = "/tmp/_git_askpass.sh"
with open(askpass, "w") as f:                       # token never enters argv or cell output
    f.write("#!/bin/sh\ncase \"$1\" in *sername*) echo x-access-token ;; *) echo %s ;; esac\n"
            % GITHUB_TOKEN)
os.chmod(askpass, 0o700)
env = dict(os.environ, GIT_ASKPASS=askpass, GIT_TERMINAL_PROMPT="0")
url = f"https://github.com/{REPO}.git"

if not os.path.isdir(os.path.join(PKG, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", url, PKG], check=True, env=env)
else:
    subprocess.run(["git", "fetch", "origin", "main"], cwd=PKG, check=True, env=env)
    subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=PKG, check=True, env=env)
os.chdir(PKG); sys.path.insert(0, PKG)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "datasets", "soundfile", "av"], check=True)
print(subprocess.check_output(["git", "log", "--oneline", "-1"], text=True).strip())

In [ ]:
# ---- invariants: must print PIPELINE CHECK OK ----
import run_omni
run_omni.main(["check", "--preset", "md", "--base-len", "64"])

In [ ]:
# ---- smoke: 4 steps, both arms, ~2 min. Tells you the real stream rate ----
import subprocess, sys
r = subprocess.run([sys.executable, "-u", "run_omni.py", "train", "--arms", "strict,ar",
                    "--preset", "md", "--base-len", "64", "--batch", "16", "--steps", "4",
                    "--holdout-mult", "4", "--live", "--log-every", "1"], cwd="/content/ncm_omni")
print("exit", r.returncode, "(0 = it saved and left cleanly; a hang here is a bug, not patience)")

In [ ]:
# ---- the run: 30,000 samples, checkpoints every 200 steps, then the eval table ----
# subprocess.run() can swallow a child's stdout until the cell ends; this streams it line by
# line as it lands, with an elapsed clock, so silence is never ambiguous.
import os, subprocess, sys, time

PKG = "/content/ncm_omni"

def live(argv, head=""):
    print(f"{head}[{time.strftime('%H:%M:%S')}]", " ".join(argv), flush=True)
    t0 = time.time()
    p = subprocess.Popen([sys.executable, "-u", "run_omni.py"] + argv,
                         cwd=PKG, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         bufsize=1)
    n = 0
    for line in p.stdout:
        n += 1
        print(f"{time.strftime('%H:%M:%S')} (+{time.time()-t0:5.0f}s) {line.rstrip()}", flush=True)
    p.wait()
    print(f"[{time.strftime('%H:%M:%S')}] child exit {p.returncode}, {n} lines in {(time.time()-t0)/60:.1f} min", flush=True)
    return p.returncode

live(["train", "--arms", "strict,ar", "--preset", "md", "--base-len", "64",
      "--batch", "16", "--steps", "1875", "--holdout-mult", "4",
      "--live", "--save-every", "200", "--log-every", "1"], head="TRAIN start\n")

for arm in ("strict", "ar"):
    f = os.path.join(PKG, f"run/train_{arm}/final.pt")
    print(arm, "final.pt:", os.path.exists(f),
          f"({round(os.path.getsize(f)/1e6,1)} MB)" if os.path.exists(f) else "", flush=True)

live(["eval", "--preset", "md", "--base-len", "64"], head="EVAL start\n")


In [ ]:
# ---- read the eval table back if it was written ----
import glob, json, os
for f in sorted(glob.glob("/content/ncm_omni/run/eval*.json")):
    print("==", os.path.basename(f))
    d = json.load(open(f))
    print(json.dumps(d, indent=1)[:2000])

## Reading the result

Only the **conditional** numbers carry a claim; free-running is structurally void (feeding the model
its own codes compounds one bad level into every coarser one). Compare per modality, never the mean.

At 0.18 positions/parameter both arms are undertrained, so a gap here is a direction, not a verdict.
A real kill test needs ~1.0, which the live path cannot reach inside one session (31 h at the
measured 1.5 samples/s): that requires the 62-minute produce step and duplex-from-shards.